In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Getting Started: Meta Muse Spark 1.3 on Gemini Enterprise Agent Platform
<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/official/generative_ai/meta_muse_intro.ipynb">
      <img src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fvertex-ai-samples%2Fmain%2Fnotebooks%2Fofficial%2Fgenerative_ai%2Fmeta_muse_intro.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/notebooks/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/vertex-ai-samples/main/notebooks/official/generative_ai/meta_muse_intro.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/official/generative_ai/meta_muse_intro.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

## Overview

### Meta Muse Spark 1.3 on Gemini Enterprise Agent Platform

Meta Muse models are available as a managed Model-as-a-Service (MaaS) endpoint on the **Gemini platform** through Google Cloud. This integration combines Meta's state-of-the-art model capabilities with Google Cloud's enterprise-grade security, scalability, and operational management. This offering provides a unified, secure, and familiar developer experience:

* **Identity and Access Management:** Authenticate using existing **Google Cloud IAM** roles and service accounts, eliminating the need to provision, rotate, or manage separate API keys.
* **API Compatibility:** Model API is drop-in compatible with the **OpenAI SDK** (`publishers/meta/v1/responses`), and OpenAI-compatible agent CLIs, so most stacks work with a base-URL and key change.
* **Unified Billing:** All model consumption appears directly on your **unified Google Cloud invoice**.
* **Security and Enterprise Data Privacy:** All execution takes place within **Google Cloud's security perimeter**. Your prompts, context, and generated responses are strictly protected and **never used to train underlying models**.

### Available Meta Muse models & Preview features

Note: For the latest information on what is supported, please refer to the [Partner Models Overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/partner-models/use-partner-models) and the Meta Muse Spark 1.3 model card in Model Garden.

#### Meta Muse Spark 1.3 (`muse-spark-1.3`)
Meta's Mixture-of-Experts reasoning model, built for long-horizon agentic work, tool use, and coding, with a 1M token context window.

### Prerequisites

Before you begin, ensure you have the following:

**Google Cloud Resources:**
* **Google Cloud Project:** A [Google Cloud Project](https://cloud.google.com/resource-manager/docs/creating-managing-projects) allowlisted for the Meta Muse 1.3 model. If your project is not allowlisted, contact your Google Cloud sales representative before proceeding.
* **Enabled APIs:** [Gemini Enterprise Agent Platform API (`aiplatform.googleapis.com`)](https://console.cloud.google.com/apis/enableflow?apiid=aiplatform.googleapis.com).
* **Model Garden Enablement:** Meta Muse 1.3 model enabled in Model Garden.
* **IAM Permissions:** [Agent Platform User (`roles/aiplatform.user`)](https://cloud.google.com/iam/docs/roles-permissions/aiplatform).

**Local Development Environment:**
* Python 3.10+ installed
* [gcloud CLI installed](https://cloud.google.com/sdk/docs/install-sdk) and [authenticated](https://cloud.google.com/docs/authentication/set-up-adc-local-dev-environment)

## Objective

This notebook shows how to:
1. Enable Meta Muse Spark 1.3 (`muse-spark-1.3`) in Model Garden and configure `vertexai.allowedModels` Organization Policies (if required).
2. Make model requests to Meta Muse Spark 1.3 using **cURL** against the Gemini Enterprise Agent Platform REST API (unary and streaming).
3. Make model requests using the **OpenAI Python library** (`openai`) with Google Cloud Application Default Credentials (Option A: `google-auth` for notebooks/Colab, Option B: environment variables for shell/local scripts, unary, streaming, and multimodal image & PDF inputs).

## Get Started

### Install required packages

In [ ]:
%pip install -U -q openai google-auth requests httpx ipywidgets

### Authenticate your notebook environment

If you are running this notebook on Google Colab, run the cell below to authenticate your environment.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

If you are running locally or in a shell environment, configure Application Default Credentials (ADC) by running:
```bash
gcloud auth application-default login
```

#### Select Meta Muse model

Throughout this notebook, we use **Meta Muse Spark 1.3** (`muse-spark-1.3`).

In [ ]:
MODEL = "muse-spark-1.3"  # @param ["muse-spark-1.3"]
available_regions = ["global"]

#### Select a location

During preview, Meta Muse Spark 1.3 is available on the **Global** endpoint:
* **`global`:** `aiplatform.googleapis.com`

In [ ]:
import ipywidgets as widgets
from IPython.display import display

dropdown = widgets.Dropdown(
    options=available_regions,
    value="global",
    description="Select a location:",
    font_weight="bold",
    style={"description_width": "initial"},
)


def dropdown_eventhandler(change):
    global LOCATION
    if change["type"] == "change" and change["name"] == "value":
        LOCATION = change.new
        print("Selected:", change.new)


LOCATION = dropdown.value
dropdown.observe(dropdown_eventhandler, names="value")
display(dropdown)

#### Set Google Cloud project and endpoint information

To get started, you must have an existing Google Cloud project and [enable the Gemini Enterprise Agent Platform (Vertex AI) API](https://console.cloud.google.com/apis/enableflow?apiid=aiplatform.googleapis.com). Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
import os
import subprocess

# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
if not PROJECT_ID or PROJECT_ID in ("[your-project-id]", "(unset)"):
    raise ValueError(
        "Please set PROJECT_ID or run `gcloud config set project YOUR_PROJECT_ID`."
    )

ENDPOINT = (
    "aiplatform.googleapis.com"
    if LOCATION == "global"
    else f"aiplatform.{LOCATION}.rep.googleapis.com"
)
BASE_URL = f"https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/meta/v1"

### First Steps: Enable the Model & Configure Organization Policy

#### 1. Enable the model in Model Garden

You can enable Meta Muse Spark 1.3 in the Cloud Console **Gemini Enterprise Agent Platform Model Garden** UI (by searching for **Muse Spark 1.3**, opening the model card, and completing the enablement and agreement steps) or programmatically via the API.

In [ ]:
PUBLISHER = "meta"
ENABLE_ENDPOINT = "us-central1-aiplatform.googleapis.com"
PARENT = f"projects/{PROJECT_ID}"
NAME = f"publishers/{PUBLISHER}/models/{MODEL}"

!curl -X POST \
  -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
  -H "x-goog-user-project: {PROJECT_ID}" \
  -H "Content-Type: application/json; charset=utf-8" \
  -d '{{}}' \
  "https://{ENABLE_ENDPOINT}/v1beta1/{PARENT}/{NAME}:enableModel"

#### 2. Allowlist the model in Organization Policy (if required)

If your organization restricts model access using the [`vertexai.allowedModels`](https://cloud.google.com/gemini-enterprise-agent-platform/models/control-model-access#allow_a_set_of_models_and_deny_all_other_models) organization policy constraint, include the Meta Muse Spark 1.3 model slug in your allowlist:

* `publishers/meta/models/muse-spark-1.3:predict`

## Making Model Requests Using cURL (Gemini Enterprise Agent Platform REST API)

#### Import required libraries

In [ ]:
import base64
import json  # noqa: F401

import httpx
from IPython.display import Image

### Text generation

#### Unary call (No streaming)

Sends a synchronous `POST` request to the Meta Muse Responses API endpoint (`publishers/meta/v1/responses`). A successful request returns a completed response object with output text.

In [ ]:
PAYLOAD = {
    "model": MODEL,
    "input": "Who are you?",
}

request = json.dumps(PAYLOAD)
!curl --fail-with-body -X POST \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
  "https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/meta/v1/responses" \
  -d '{request}'

#### Streaming call (With streaming)

Sends a streaming `POST` request with `"stream": True`. Streaming requests return server-sent events (SSE), including `response.output_text.delta` events followed by `response.completed`.

In [ ]:
PAYLOAD = {
    "model": MODEL,
    "input": "Who are you?",
    "stream": True,
}

request = json.dumps(PAYLOAD)
!curl --fail-with-body --no-buffer -X POST \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $(gcloud auth application-default print-access-token)" \
  "https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/meta/v1/responses" \
  -d '{request}'

## Making Model Requests Using the OpenAI Python Library

The [OpenAI Python library](https://github.com/openai/openai-python) (`openai`) provides convenient access to Meta Muse Spark 1.3 on Google Cloud from any Python 3.10+ application.

### Option A: Use OpenAI Python library to make model requests (Recommended for Notebooks / Colab environments)

Use `google.auth` to obtain Application Default Credentials (ADC) and pass the bearer token and Gemini Enterprise Agent Platform `base_url` (`publishers/meta/v1`) to `OpenAI(...)`.

> **Note:** ADC access tokens expire in ~1 hour. If you get `401` errors (e.g. in long-running jobs), call `creds.refresh(google.auth.transport.requests.Request())` again and create a new client (or set `client.api_key = creds.token`).

In [ ]:
import google.auth
import google.auth.transport.requests
from openai import OpenAI

ENDPOINT = (
    "aiplatform.googleapis.com"
    if LOCATION == "global"
    else f"aiplatform.{LOCATION}.rep.googleapis.com"
)

creds, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
creds.refresh(google.auth.transport.requests.Request())

client = OpenAI(
    base_url=f"https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/meta/v1",
    api_key=creds.token,
)

### Text generation

#### Without Streaming (Unary call)

Sends a synchronous request via `client.responses.create` and prints the response.

In [ ]:
# Without Streaming
response = client.responses.create(
    model=MODEL,
    input="Who are you?",
    store=False,
)
print(response)

#### With Streaming (Streaming call)

Sends a streaming request via `client.responses.create(..., stream=True)` and prints each event as it arrives.

In [ ]:
# With Streaming
stream = client.responses.create(
    model=MODEL,
    input="Who are you?",
    stream=True,
)
for event in stream:
    print(event)

### Multimodal input (Image and PDF documents)

Meta Muse Spark 1.3 supports text, image, and PDF document inputs with up to a 1M token context window.

#### Encode and preview image

We fetch a sample image from the public `cloud-samples-data` Cloud Storage bucket using `httpx`, encode it as base64, and preview it inline.

In [ ]:
image_url = (
    "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/scones.jpg"
)
image_media_type = "image/jpeg"
image_bytes = httpx.get(image_url).content
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

Image(image_bytes, width=300, height=200)

#### Multimodal call with image and text

Sends a base64-encoded image (`input_image`) alongside a text prompt (`input_text`) using the Responses API.

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Describe this image."},
                {
                    "type": "input_image",
                    "image_url": f"data:{image_media_type};base64,{image_b64}",
                },
            ],
        }
    ],
    store=False,
)
print(response.output_text)

#### Multimodal call with PDF document and text

Fetches a sample PDF document from `cloud-samples-data`, encodes it as base64, and sends it (`input_file`) alongside a text prompt (`input_text`).

In [ ]:
pdf_url = (
    "https://storage.googleapis.com/cloud-samples-data/generative-ai/pdf/2403.05530.pdf"
)
pdf_bytes = httpx.get(pdf_url).content
pdf_b64 = base64.b64encode(pdf_bytes).decode("utf-8")

response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Provide a brief 3-bullet summary of this PDF document.",
                },
                {
                    "type": "input_file",
                    "filename": "document.pdf",
                    "file_data": f"data:application/pdf;base64,{pdf_b64}",
                },
            ],
        }
    ],
    store=False,
)
print(response.output_text)

### Structured outputs via JSON Schema

Meta Muse Spark 1.3 supports structured outputs conforming to a provided JSON Schema.

In [ ]:
response = client.responses.create(
    model=MODEL,
    input="Extract the model name, provider, and max input context window from: Meta Muse Spark 1.3 offers up to 1M input tokens on Google Cloud.",
    text={
        "format": {
            "type": "json_schema",
            "name": "model_specs",
            "schema": {
                "type": "object",
                "properties": {
                    "model_name": {"type": "string"},
                    "provider": {"type": "string"},
                    "max_context_tokens": {"type": "string"},
                },
                "required": ["model_name", "provider", "max_context_tokens"],
                "additionalProperties": False,
            },
            "strict": True,
        }
    },
    store=False,
)
print(response.output_text)

### Option B: Use OpenAI Python library to make model requests (Recommended for Shell / Local Scripts)

In shell scripts or local applications, you can export `OPENAI_BASE_URL` and `OPENAI_API_KEY` so `OpenAI()` picks them up automatically from environment variables:

```bash
export PROJECT_ID="your-project-id"
export LOCATION="global"
export OPENAI_BASE_URL="https://aiplatform.googleapis.com/v1/projects/${PROJECT_ID}/locations/${LOCATION}/publishers/meta/v1"
export OPENAI_API_KEY="$(gcloud auth application-default print-access-token)"
```

> **Note:** Run the `export` commands and Python script in the same shell environment. When the token expires (~1 hour), re-run the `OPENAI_API_KEY` export and restart the script, or update the running client's `api_key` with a refreshed token.

In [ ]:
import os

from openai import OpenAI

os.environ["OPENAI_BASE_URL"] = (
    f"https://{ENDPOINT}/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/meta/v1"
)
os.environ["OPENAI_API_KEY"] = creds.token

client = (
    OpenAI()
)  # picks up OPENAI_BASE_URL and OPENAI_API_KEY from environment variables
response = client.responses.create(
    model=MODEL,
    input="Who are you?",
)
print(response)

## Error Handling & Other Helpful Information

### Error handling

A reasonable default is exponential backoff with jitter on `429` and `5xx`, capped at a handful of attempts.

| Status | Meaning | Recommended action |
| :---: | :--- | :--- |
| `401` | Token expired or invalid | Refresh the access token and retry |
| `403` | Identity lacks `roles/aiplatform.user`, or the project or caller is not enabled | Verify IAM bindings and that the model is enabled on the project |
| `404` | Endpoint path, location, or model identifier incorrect | Check the URL structure and location segment, and verify that the model ID is `muse-spark-1.3` |
| `429` | Rate or quota limit reached | Back off exponentially and retry; contact your account team if sustained |
| `5xx` | Transient backend condition | Retry with exponential backoff |